In [1]:
!pip install -q langgraph langchain-openai langchain-chroma langchain-huggingface sentence-transformers langchain-community

In [2]:
import os
import getpass
from typing import List, TypedDict
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END

In [3]:
# ==========================================
# 1. 환경 설정 및 DB 로드
# ==========================================
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

print("--- 임베딩 모델 및 Vector DB 로드 중... ---")
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="/content/drive/MyDrive/kt_cs_agent/chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

# ==========================================
# 2. 문서 관리 대장 (Document Registry)
# 여기서는 미리 선언하는 딕셔너리 구조로 작성됨(간단한 서버 구조에서는 db에서 리스트 가져와서 사용할 것임)
# ==========================================
DOC_REGISTRY = {
    "인터넷이용약관": "/content/(이용약관전문)인터넷서비스이용약관_202509.pdf",
    "TV서비스약관": "/content/TV서비스이용약관.pdf",      # (예시: 실제 파일 없으면 필터링 안됨)
    "모바일이용약관": "/content/모바일서비스이용약관.pdf"   # (예시)
}

# 프롬프트에 보여줄 목록 문자열 생성
DOC_LIST_STR = "\n".join([f"- {name}" for name in DOC_REGISTRY.keys()])

# ==========================================
# 3. LangGraph 상태(State) 정의
# ==========================================
class SearchState(TypedDict):
    summary: str            # [입력] 상담 요약
    target_doc_name: str    # [중간] 선택된 문서 별칭 (예: 인터넷이용약관)
    search_query: str       # [중간] 추출된 검색 키워드
    documents: List[Document] # [결과] 검색된 문서 리스트

# ==========================================
# 4. 노드(Node) 정의
# ==========================================

# [Node 1] 상담 내용을 보고 문서 선택(Routing) 및 키워드 추출(Querying)
def analyzer_node(state: SearchState):
    summary = state["summary"]
    print(f"\n🔍 [1. 분석] 상담 내용 분석 및 문서 라우팅 중...")

    llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

    prompt = ChatPromptTemplate.from_template(
        """
        당신은 상담 내용을 분석하여 검색 전략을 수립하는 관리자입니다.

        [보유 문서 목록]
        {doc_list}

        [상담 요약]
        {summary}

        위 내용을 바탕으로 가장 연관된 '문서 이름(목록 중 택1)'과 '검색 키워드'를 결정하세요.
        문서를 특정하기 어려우면 문서 이름에 '없음'이라고 적으세요.

        출력 형식: 문서이름 | 검색키워드
        예시: 인터넷이용약관 | 해지 위약금 산정식
        """
    )

    chain = prompt.partial(doc_list=DOC_LIST_STR) | llm | StrOutputParser()
    response = chain.invoke({"summary": summary})

    # 결과 파싱
    try:
        parts = response.split("|")
        target_name = parts[0].strip()
        query = parts[1].strip()
    except:
        target_name = "없음"
        query = response

    print(f"👉 전략 수립: 문서[{target_name}] / 키워드[{query}]")

    return {"target_doc_name": target_name, "search_query": query}

# [Node 2] 하이브리드 검색 수행 (Scoped + Global)
def search_node(state: SearchState):
    target_name = state["target_doc_name"]
    query = state["search_query"]
    docs = []

    print(f"📚 [2. 검색] 하이브리드 검색 수행 중...")

    # 전략 1: 타겟 문서 집중 검색 (2개) - Scoped
    if target_name in DOC_REGISTRY:
        real_path = DOC_REGISTRY[target_name] # 별칭 -> 실제 경로 변환
        print(f"   - [Scoped] '{target_name}'(경로 매핑됨) 내부 검색 (k=2)")

        # 실제 파일 경로로 메타데이터 필터링
        try:
            scoped_results = vectorstore.similarity_search(
                query,
                k=2,
                filter={"source": real_path}
            )
            docs.extend(scoped_results)
        except Exception as e:
            print(f"   ⚠️ 필터링 검색 중 오류 발생 (파일 경로 확인 필요): {e}")
    else:
        print("   - [Scoped] 특정된 문서가 없어 집중 검색을 건너뜁니다.")

    # 전략 2: 전체 범위 검색 (1개) - Global (보험용)
    print("   - [Global] 전체 문서 대상 보완 검색 (k=1)")
    global_results = vectorstore.similarity_search(query, k=1)
    docs.extend(global_results)

    # 중복 제거 (내용 기반)
    unique_docs = []
    seen_content = set()
    for doc in docs:
        # lcel 버전과 다르게(lcel버전은 전체 일치) 내용의 앞 50자만 비교해서 중복 체크
        content_hash = doc.page_content[:50]
        if content_hash not in seen_content:
            unique_docs.append(doc)
            seen_content.add(content_hash)

    print(f"✅ 총 {len(docs)}개의 유니크한 문서가 검색되었습니다.")
    return {"documents": unique_docs}

# ==========================================
# 5. 그래프(Graph) 구성
# ==========================================
workflow = StateGraph(SearchState)

workflow.add_node("analyzer", analyzer_node)
workflow.add_node("searcher", search_node)

workflow.add_edge(START, "analyzer")
workflow.add_edge("analyzer", "searcher")
workflow.add_edge("searcher", END)

app = workflow.compile()

# ==========================================
# 6. 결과 포맷팅 및 실행 함수
# ==========================================
def format_docs(docs: List[Document]):
    output = ""
    for i, doc in enumerate(docs):
        source_raw = doc.metadata.get("source", "Unknown")
        source_name = source_raw.split("/")[-1]
        page = doc.metadata.get("page", 0) + 1

        output += f"""
        📄 [문서 {i+1}] {source_name} (p.{page})
        ────────────────────────────────────────
        {doc.page_content[:200].replace('\n', ' ')}...
        ────────────────────────────────────────
        """
    return output

def run_search_system(summary_input):
    print("\n" + "="*60)
    print(f"🚀 [시스템 시작] 입력: {summary_input}")
    print("="*60)

    inputs = {"summary": summary_input}
    result = app.invoke(inputs)

    print("\n" + "="*60)
    print(f"🗝️ 최종 키워드: {result['search_query']}")
    print("-" * 60)
    print(format_docs(result['documents']))
    print("=" * 60)

# 테스트 실행
run_search_system("인터넷 약정 해지 시 위약금 계산법이 궁금합니다.")

OpenAI API Key를 입력하세요: ··········
--- 임베딩 모델 및 Vector DB 로드 중... ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



🚀 [시스템 시작] 입력: 인터넷 약정 해지 시 위약금 계산법이 궁금합니다.

🔍 [1. 분석] 상담 내용 분석 및 문서 라우팅 중...
👉 전략 수립: 문서[인터넷이용약관] / 키워드[해지 위약금 산정식]
📚 [2. 검색] 하이브리드 검색 수행 중...
   - [Scoped] '인터넷이용약관'(경로 매핑됨) 내부 검색 (k=2)
   - [Global] 전체 문서 대상 보완 검색 (k=1)
✅ 총 3개의 유니크한 문서가 검색되었습니다.

🗝️ 최종 키워드: 해지 위약금 산정식
------------------------------------------------------------

        📄 [문서 1] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.116)
        ────────────────────────────────────────
        ※ 계약기간 이내 해지시는 할인받은 요금 및 단말장치사용료를 반환하여야 합니다.    - 이용료 산정식 = 할인금액 ⅹ 경과월수 ⅹ(1-사용기간 할인율/계약기간 할인율)     · 사용기간 할인율은 2년 미만은 1년 계약, 3년 미만은 2년 계약, 4년 미만은 3년 계약의 할인율을 적용    - 단말장치사용료 산정식은 요금표 3-가-(3)장비임대료 할인액 ...
        ────────────────────────────────────────
        
        📄 [문서 2] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.167)
        ────────────────────────────────────────
        -  167 -    ※ 계약기간 이내 AP를 해지하거나 계약기간을 단축할 시는 추가 할인된 요금을 반환해야 하며,     할인반환금은 아래의 산식을 적용 받습니다.     - 산정식 : ∑ [약정기간별 총 할인금액 X (1 – 